In [0]:
from pyspark.sql import SparkSession
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.functions import trim
from pyspark.sql.functions import regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, FloatType
from datetime import datetime


In [0]:
def get_month():
    today = datetime.now()
    actual_month = today.month 
    actual_year = today.year

    if today.month == 1:

        actual_month = 12
        actual_year = today.year - 1
        url_date = f'{actual_year}-{actual_month}'
    else:
        actual_month = today.month - 1
        url_date = f'{actual_year}-{actual_month:02d}'
        return(url_date)

url_date = get_month()

In [0]:
spark = SparkSession.builder \
    .appName("enterprises_silver") \
    .getOrCreate()
       

enterprises_schema = StructType ([ 
    StructField("DocumentNumber", StringType(), True),      
    StructField("BusinessName", StringType(), True),   
    StructField("LegalNature", StringType(), False),   
    StructField("Qualification", StringType(), True),     
    StructField("ShareCapital", FloatType(), True),       
    StructField("CompanySize", StringType(), True),       
    StructField("Federative entity", StringType(), True)])

url_date = get_month()

In [0]:
df_enterprises = spark.read \
    .format("csv")\
    .option("delimiter",';')\
    .schema(enterprises_schema) \
    .load(f'/Volumes/workspace/cnpjs_schema/establishments/{url_date}_data/bronze').select([
        'DocumentNumber',
        'BusinessName',
        'LegalNature',
        'CompanySize',
        'ShareCapital',
    ])



In [0]:
legal_natures_codes = pd.read_csv('/Volumes/workspace/cnpjs_schema/auxiliar/legal_natures/2025-12_Naturezas.csv', 
    delimiter=';', 
    encoding='latin-1',
    names = ['Code', 'Legal_Nature'],
    dtype = {'Code': 'str', 'Legal_Nature': 'str'},
    index_col=['Code'])
legal_natures_dict = legal_natures_codes.to_dict()
legal_natures_dict['Legal_Nature']

In [0]:
df_enterprises = df_enterprises.withColumn('BusinessName', 
    F.trim(
        F.regexp_replace(

            F.col('BusinessName'), 
            r"[^a-zA-Z0-9\s]", 
            ""
        )
    )
)
df_enterprises = df_enterprises.withColumn('ShareCapital', 
    F.regexp_replace(
    F.col('ShareCapital'),",", 
    "."
    )
)

df_enterprises = df_enterprises.withColumn('ShareCapital', F.trim(F.col("DocumentNumber")))
df_enterprises = df_enterprises.withColumn('ShareCapital', df_enterprises['ShareCapital'].try_cast(DoubleType()))

df_enterprises = df_enterprises.replace(legal_natures_dict['Legal_Nature'],subset=['LegalNature'])


df_enterprises.display()

In [0]:
df_enterprises.write.mode('overwrite').format('delta').saveAsTable('workspace.enterprises_silver.silver_enterprise')